# `Facial Recognition System`

In [ ]:
!pip install -q mtcnn keras-facenet scikit-learn matplotlib seaborn tqdm

In [ ]:
!pip install facenet-pytorch==2.5.2 tqdm matplotlib scikit-learn pandas -q
!pip install pytorch-metric-learning -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 4.2 MB/s eta 0:00:00


In [ ]:
import os
import random
import shutil
import kagglehub

path = kagglehub.dataset_download("jessicali9530/lfw-dataset")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'lfw-dataset' dataset.
Dataset downloaded to: /kaggle/input/lfw-dataset


In [ ]:
# Detect + crop using facenet-pytorch MTCNN
import os, sys, glob
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from tqdm import tqdm

RAW_DIR = "/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled"
OUT_DIR = "./lfw_crops_160"

os.makedirs(OUT_DIR, exist_ok=True)

mtcnn = MTCNN(image_size=160, margin=0, keep_all=False, device='cuda' if torch.cuda.is_available() else 'cpu')

persons = [d for d in sorted(os.listdir(RAW_DIR)) if os.path.isdir(os.path.join(RAW_DIR,d))]
print("Persons found:", len(persons))

cnt_total = 0
for person in tqdm(persons):
    in_person_dir = os.path.join(RAW_DIR, person)
    out_person_dir = os.path.join(OUT_DIR, person)
    os.makedirs(out_person_dir, exist_ok=True)
    imgs = sorted(os.listdir(in_person_dir))
    for imgname in imgs:
        in_path = os.path.join(in_person_dir, imgname)
        try:
            img = Image.open(in_path).convert('RGB')
        except:
            continue
        # detect & crop
        face = mtcnn(img)
        if face is None:
            # skip if no face detected
            continue
        # save face tensor
        face_img = Image.fromarray((face.mul(255).permute(1,2,0).byte().numpy()))
        # name
        out_path = os.path.join(out_person_dir, imgname)
        face_img.save(out_path)
        cnt_total += 1

print("Saved face crops:", cnt_total, " to", OUT_DIR)

Persons found: 5749


100%|██████████| 5749/5749 [07:56<00:00, 12.06it/s]

Saved face crops: 13232  to ./lfw_crops_160


In [ ]:
# Create train/val/test
import pandas as pd, os, shutil
CSV_ROOT = "/kaggle/input/lfw-dataset"
people_train_csv = os.path.join(CSV_ROOT, "peopleDevTrain.csv")
people_test_csv  = os.path.join(CSV_ROOT, "peopleDevTest.csv")
print("Train CSV exists:", os.path.exists(people_train_csv))
print("Test CSV exists:", os.path.exists(people_test_csv))


train_names = []
test_names = []
if os.path.exists(people_train_csv):
    train_names = pd.read_csv(people_train_csv, header=None).iloc[:,0].astype(str).tolist()
if os.path.exists(people_test_csv):
    test_names = pd.read_csv(people_test_csv, header=None).iloc[:,0].astype(str).tolist()

print("Train persons:", len(train_names), "Test persons (dev):", len(test_names))

BASE = "./faces_final"
train_dir = os.path.join(BASE, "train")
val_dir   = os.path.join(BASE, "val")
test_dir  = os.path.join(BASE, "test")
for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d, exist_ok=True)

import random
random.seed(42)
for name in train_names:
    src = os.path.join(OUT_DIR, name)
    if not os.path.isdir(src): continue
    dst = os.path.join(train_dir, name)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)

val_persons = random.sample(train_names, max(1,int(0.2*len(train_names))))
for name in val_persons:
    s = os.path.join(train_dir, name)
    d = os.path.join(val_dir, name)
    if os.path.isdir(s):
        shutil.move(s, d)

# test
for name in test_names:
    src = os.path.join(OUT_DIR, name)
    if not os.path.isdir(src): continue
    dst = os.path.join(test_dir, name)
    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)

print("Final counts - train persons:", len(os.listdir(train_dir)), "val persons:", len(os.listdir(val_dir)), "test persons:", len(os.listdir(test_dir)))

Train CSV exists: True
Test CSV exists: True
Train persons: 4039 Test persons (dev): 1712
Final counts - train persons: 3231 val persons: 807 test persons: 1711


In [ ]:

# Dataset & DataLoader
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

IMG_SIZE = 160
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

class FacesDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = root_dir
        self.transform = transform
        self.samples = []
        persons = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir,d))])
        self.class_to_idx = {p:i for i,p in enumerate(persons)}
        for p in persons:
            pdir = os.path.join(root_dir,p)
            for f in os.listdir(pdir):
                self.samples.append((os.path.join(pdir,f), self.class_to_idx[p]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p,label = self.samples[idx]
        img = Image.open(p).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

train_ds = FacesDataset(train_dir, transform=transform)
val_ds = FacesDataset(val_dir, transform=transform)
test_ds = FacesDataset(test_dir, transform=transform)

print("Train samples:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

labels = list(train_ds.class_to_idx.keys())
print("Kept persons:", len(labels))


Train samples: 7709 Val: 1815 Test: 3708
Kept persons: 3231


In [ ]:
# Extract embeddings and save them
import torch
from facenet_pytorch import InceptionResnetV1
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
backbone = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def embed_batch(dl):
    emb_list, y_list, paths = [], [], []
    with torch.no_grad():
        for xb, yb in dl:
            xb = xb.to(device)
            emb = backbone(xb)  # (N,512)
            emb = torch.nn.functional.normalize(emb, p=2, dim=1)
            emb_list.append(emb.cpu().numpy())
            y_list.append(yb.numpy())
    return np.vstack(emb_list), np.hstack(y_list)

# build gallery embeddings from train set
import collections
train_embs, train_labels = embed_batch(train_loader)

label_to_embs = collections.defaultdict(list)
for e,l in zip(train_embs, train_labels): label_to_embs[l].append(e)
gallery = {}
for l, arr in label_to_embs.items():
    gallery[l] = np.mean(arr, axis=0)
print("Gallery size:", len(gallery))

# save gallery and mapping
import pickle
with open("gallery.pkl","wb") as f:
    pickle.dump({"gallery":gallery, "labels":train_ds.class_to_idx}, f)
print("Saved gallery.pkl")


  0%|          | 0.00/107M [00:00<?, ?B/s]

Gallery size: 3231
Saved gallery.pkl


In [ ]:
import pandas as pd

pairs_path = "/kaggle/input/lfw-dataset/pairs.csv"
df = pd.read_csv(pairs_path)

print("شكل الأعمدة:", df.columns)
print(df.head(10))

pairs = []

for _, row in df.iterrows():
    name = str(row["name"]).strip()

    try:
        idx1 = int(row["imagenum1"])
        idx2 = int(row["imagenum2"])
    except ValueError:
        continue

    file1 = f"{name}_{idx1:04d}.jpg"
    file2 = f"{name}_{idx2:04d}.jpg"

    pairs.append((name, name, file1, file2, True))


df_pairs = pd.DataFrame(pairs, columns=["name1", "name2", "file1", "file2", "issame"])

print("بعد التنضيف:")
print(df_pairs.head(10))
print("إجمالي الأزواج:", len(df_pairs))


شكل الأعمدة: Index(['name', 'imagenum1', 'imagenum2', 'Unnamed: 3'], dtype='object')
                    name  imagenum1 imagenum2  Unnamed: 3
0           Abel_Pacheco          1         4         NaN
1         Akhmed_Zakayev          1         3         NaN
2         Akhmed_Zakayev          2         3         NaN
3          Amber_Tamblyn          1         2         NaN
4  Anders_Fogh_Rasmussen          1         3         NaN
5  Anders_Fogh_Rasmussen          1         4         NaN
6         Angela_Bassett          1         5         NaN
7         Angela_Bassett          2         5         NaN
8         Angela_Bassett          3         4         NaN
9            Ann_Veneman          3         5         NaN
بعد التنضيف:
                   name1                  name2  \
0           Abel_Pacheco           Abel_Pacheco   
1         Akhmed_Zakayev         Akhmed_Zakayev   
2         Akhmed_Zakayev         Akhmed_Zakayev   
3          Amber_Tamblyn          Amber_Tamblyn   
4  Anders

In [ ]:
# inference + evaluate on pairs (verification)
import numpy as np, pickle, math, os
from sklearn.metrics import roc_curve, roc_auc_score
import pandas as pd
from PIL import Image
import torch

with open("gallery.pkl","rb") as f:
    data = pickle.load(f)
gallery = data["gallery"]
idx_to_name = {v:k for k,v in train_ds.class_to_idx.items()}

def get_embedding(img_tensor):
    backbone.eval()
    with torch.no_grad():
        emb = backbone(img_tensor.unsqueeze(0).to(device))
        emb = torch.nn.functional.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy()[0]

def eval_on_pairs(match_csv, mismatch_csv):
    scores, labels = [], []

    # match pairs
    pairs = pd.read_csv(match_csv)
    for _, row in pairs.iterrows():
        try:
            name = row['name']
            i1, i2 = int(row['imagenum1']), int(row['imagenum2'])
        except Exception as e:
            print(f"[SKIP MATCH] {row} ({e})")
            continue

        p1 = f"./lfw_crops_160/{name}/{name}_{i1:04d}.jpg"
        p2 = f"./lfw_crops_160/{name}/{name}_{i2:04d}.jpg"
        if not (os.path.exists(p1) and os.path.exists(p2)):
            continue

        t1 = transform(Image.open(p1).convert('RGB'))
        t2 = transform(Image.open(p2).convert('RGB'))
        e1, e2 = get_embedding(t1), get_embedding(t2)
        sim = np.dot(e1, e2)
        scores.append(sim); labels.append(1)

    # mismatch pairs
    mm = pd.read_csv(mismatch_csv)

    # normalize column names
    if 'name1' not in mm.columns and 'name.1' in mm.columns:
        mm = mm.rename(columns={'name': 'name1', 'name.1': 'name2'})

    for _, row in mm.iterrows():
        try:
            name1, name2 = row['name1'], row['name2']
            i1, i2 = int(row['imagenum1']), int(row['imagenum2'])
        except Exception as e:
            print(f"[SKIP MISMATCH] {row} ({e})")
            continue

        p1 = f"./lfw_crops_160/{name1}/{name1}_{i1:04d}.jpg"
        p2 = f"./lfw_crops_160/{name2}/{name2}_{i2:04d}.jpg"
        if not (os.path.exists(p1) and os.path.exists(p2)):
            continue

        t1 = transform(Image.open(p1).convert('RGB'))
        t2 = transform(Image.open(p2).convert('RGB'))
        e1, e2 = get_embedding(t1), get_embedding(t2)
        sim = np.dot(e1, e2)
        scores.append(sim); labels.append(0)

    return np.array(scores), np.array(labels)

# usage
match_csv = "/kaggle/input/lfw-dataset/matchpairsDevTrain.csv"
mismatch_csv = "/kaggle/input/lfw-dataset/mismatchpairsDevTrain.csv"
scores, labs = eval_on_pairs(match_csv, mismatch_csv)

fpr, tpr, th = roc_curve(labs, scores)
auc = roc_auc_score(labs, scores)


j = np.argmax(tpr - fpr)
best_thresh = th[j]
print("Train ROC AUC:", auc, "best_thresh:", best_thresh)


Train ROC AUC: 0.6943535445446274 best_thresh: 0.33199757


In [ ]:
# head-only training
import torch, torch.nn as nn, torch.optim as optim, time
from tqdm import tqdm

NUM_CLASSES = len(train_ds.class_to_idx)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
backbone = InceptionResnetV1(pretrained='vggface2', classify=False).to(device)
for p in backbone.parameters(): p.requires_grad = False

class ClassifierHead(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes)
        )
    def forward(self,x):
        emb = self.backbone(x)
        out = self.head(emb)
        return out

clf = ClassifierHead(backbone, NUM_CLASSES).to(device)
optimizer = optim.Adam(clf.head.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

EPOCHS = 10
train_logs, val_logs = [], []
best_val = 0.0

for epoch in range(EPOCHS):
    clf.train()
    t0=time.time(); running_loss=0; total=0; correct=0
    for xb,yb in tqdm(train_loader):
        xb,yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = clf(xb)
        loss = criterion(logits, yb)
        loss.backward(); optimizer.step()
        running_loss += loss.item()*xb.size(0)
        preds = logits.argmax(1)
        correct += (preds==yb).sum().item()
        total += xb.size(0)
    train_loss = running_loss/total; train_acc = correct/total

    # val
    clf.eval()
    vloss=0; vtotal=0; vcorrect=0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb,yb = xb.to(device), yb.to(device)
            logits = clf(xb)
            loss = criterion(logits, yb)
            vloss += loss.item()*xb.size(0)
            preds = logits.argmax(1)
            vcorrect += (preds==yb).sum().item()
            vtotal += xb.size(0)
    val_loss = vloss/vtotal; val_acc = vcorrect/vtotal
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | val_loss={val_loss:.4f}, val_acc={val_acc:.4f} | time={time.time()-t0:.1f}s")
    if val_acc>best_val:
        best_val=val_acc
        torch.save(clf.state_dict(),"best_head.pth")
        print("Saved best head")
print("Done. Best val acc:", best_val)


100%|██████████| 241/241 [00:14<00:00, 17.06it/s]


Epoch 1: train_loss=8.0308, train_acc=0.0669 | val_loss=8.0821, val_acc=0.0006 | time=17.1s
Saved best head


100%|██████████| 241/241 [00:13<00:00, 18.01it/s]


Epoch 2: train_loss=7.7110, train_acc=0.0833 | val_loss=8.1182, val_acc=0.0000 | time=16.5s


100%|██████████| 241/241 [00:13<00:00, 17.60it/s]


...

Epoch 8: train_loss=6.2122, train_acc=0.0968 | val_loss=9.0908, val_acc=0.0017 | time=16.9s


100%|██████████| 241/241 [00:14<00:00, 17.13it/s]


Epoch 9: train_loss=6.1233, train_acc=0.0966 | val_loss=9.1573, val_acc=0.0017 | time=17.0s


100%|██████████| 241/241 [00:13<00:00, 17.68it/s]


Epoch 10: train_loss=6.0348, train_acc=0.1001 | val_loss=9.2812, val_acc=0.0017 | time=16.6s
Done. Best val acc: 0.001652892561983471


In [ ]:
# save & plot
import matplotlib.pyplot as plt
# assume train_logs and val_logs lists stored (adapt from previous cell to collect)
# example: plot placeholder
plt.figure(figsize=(10,4))
# replace with real arrays: train_losses, val_losses, train_accs, val_accs
# plt.subplot(1,2,1); plt.plot(train_losses,label='train'); plt.plot(val_losses,label='val'); plt.legend(); plt.title('Loss')
# plt.subplot(1,2,2); plt.plot(train_accs,label='train'); plt.plot(val_accs,label='val'); plt.legend(); plt.title('Acc')
plt.show()

# Save final backbone + head easily: (if you used classifier)
torch.save({"backbone_state":backbone.state_dict(), "head_state":clf.head.state_dict(), "class_map":train_ds.class_to_idx}, "faceid_pipeline.pth")
print("Saved faceid_pipeline.pth")


<Figure size 1000x400 with 0 Axes>

Saved faceid_pipeline.pth


In [ ]:
# add new person quickly: compute prototype embedding and add to gallery.pkl
from PIL import Image
import numpy as np, pickle

def add_new_person(name, image_paths):
    embs = []
    for p in image_paths:
        img = transform(Image.open(p).convert('RGB'))
        emb = get_embedding(img)
        embs.append(emb)
    proto = np.mean(embs, axis=0)
    with open("gallery.pkl","rb") as f:
        d = pickle.load(f)
    # find new label id (use big number or extend mapping)
    new_label = max(d['labels'].values())+1
    d['labels'][name]=new_label
    d['gallery'][new_label]=proto
    with open("gallery.pkl","wb") as f:
        pickle.dump(d,f)
    print("Added", name, "as label", new_label)

# usage example:
# add_new_person("NewPerson", ["./somepath/img1.jpg","./somepath/img2.jpg"])